# Publish training and inference from source

Fill in the three values below: bucket, role ARN and image. This notebook uploads the current source
and publishes **both** pipelines; it does not start training or inference.
After publication, run either pipeline entirely from Studio. Optional execution
and inspection cells are in `03G_inspect_source_runs.ipynb`.


### 1. Required inputs

- Enter the bucket name, with or without s3://.
- Paste the exact IAM role ARN, including its path; it is used as both the pipeline role and the job role.
- Paste the image URI; a repository address with no version on the end also works.
- The namespace names the two pipelines and their folder in the bucket; use a different namespace for CI deployment.

In [ ]:
bucket = ""                        # bucket name, with or without s3://
role_arn = ""                      # exact IAM role ARN; used as pipeline role and job role
image_uri = ""                     # repository address, repository:tag or repository@sha256:...

namespace = "source-dev"
model_ids = None                   # Discover model folders; or select ["usecase1", "usecase2"].
batch_runtime = "transform"        # Native Batch Transform; test the image with 00I first.
                                  # "processing" runs the same predictor in a Processing job.


### 2. Load this checkout

- Find the application folder from this notebook's working directory.
- Import the same publisher that a CI build can call.
- Use the one role for the pipelines and their jobs, and `s3://<bucket>/<namespace>` as the application folder.
- Print the caller identity and selected model folders; nothing is uploaded or created yet.

In [ ]:
from pathlib import Path
import json, sys
import boto3
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-models/src").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models"), str(ROOT / "mlops-models/src")]
from mlops_models.pipelines.source import (
    source_identity, source_location, upload_source, upload_examples, build_definitions, publish,
)
from mlops_skeleton.names import discover_models
model_ids = model_ids or [folder.name for folder in discover_models()]
session = boto3.Session()
sm, s3 = session.client("sagemaker"), session.client("s3")
print(session.client("sts").get_caller_identity())
print("Source:", ROOT / "mlops-models/src/mlops_models")
for model in model_ids:
    print(model, (ROOT / "mlops-models/src/mlops_models/models" / model / "model-config.json").is_file())
bucket = bucket.strip().removeprefix("s3://").rstrip("/")
role_arn, image_uri = role_arn.strip(), image_uri.strip()
assert bucket and "/" not in bucket and role_arn and image_uri, "Fill in bucket, role_arn and image_uri."
pipeline_role_arn = job_role_arn = role_arn  # Set job_role_arn separately here if the platform provides one.
s3_root = f"s3://{bucket}/{namespace}"
print("Role for pipelines and jobs:", role_arn)
print("Will use S3 folder:", s3_root)


### 3. Deployment and platform settings

- Direct deploy uploads this working copy; reuse its name to replace source between executions.
- CI deploy requires a clean checkout and the reviewed commit supplied by CI; it uses a release/commit prefix.
- Only set VPC/KMS values when the platform supplies them. ResolveModel needs access to the SageMaker API.

In [ ]:
deploy_mode = "direct-deploy"
deploy_name = "my-review"           # Development slug, or CI release name
reviewed_commit = ""               # CI supplies this full hash; not a branch name
infrastructure = {
    "network": {
        "isolation": False,
        "subnet_ids": [],
        "security_group_ids": [],
    },
    "output_kms_key_arn": "",
    "volume_kms_key_arn": "",
}
identity = source_identity(deploy_mode, deploy_name, reviewed_commit)
source_uri = source_location(s3_root, identity)
input_root = s3_root.rstrip("/") + "/inputs"
print(identity)
print("Do not overwrite direct-deploy source while an execution is running.")


### 4. Check the chosen S3 prefix

- Write a tiny probe under the application prefix using notebook credentials.
- Read the same object back; errors remain visible. The small probe file stays and is overwritten next run.
- This does not establish the job role's S3 access; managed execution checks that separately.

In [ ]:
bucket, prefix = s3_root.removeprefix("s3://").rstrip("/").split("/", 1)
assert s3_root.startswith("s3://") and prefix, "Use a bucket plus a nonempty application prefix."
probe_key = prefix + "/checks/source-deploy.txt"
s3.put_object(Bucket=bucket, Key=probe_key, Body=b"source deployment check")
print(s3.get_object(Bucket=bucket, Key=probe_key)["Body"].read().decode())


### 5. Pin the existing image

- Resolve a tag to its ECR digest; for a repository address, use its newest tagged image. Keep a supplied digest URI.
- Use that image for both training and Processing steps.
- No Docker build, image push or package installation happens here.

HPO cannot override the image's startup command. This source route requires an image
that runs `python -m mlops_models.entrypoints train` and honors `PYTHONPATH`.
Ordinary Training and Processing can override that command; their success alone
does not confirm HPO support. Confirm the platform image before selecting UseHPO.


In [ ]:
if "@sha256:" not in image_uri:
    registry, repository = image_uri.split("/", 1)
    repository, _, tag = repository.partition(":")
    ecr = session.client("ecr")
    if tag:
        detail = ecr.describe_images(
            registryId=registry.split(".")[0], repositoryName=repository,
            imageIds=[{"imageTag": tag}],
        )["imageDetails"][0]
    else:
        # Repository address only: use its newest tagged image, as 00G section 6 does.
        images = []
        for page in ecr.get_paginator("describe_images").paginate(
                registryId=registry.split(".")[0], repositoryName=repository,
                filter={"tagStatus": "TAGGED"}):
            images.extend(page["imageDetails"])
        assert images, "That repository has no tagged images; paste a full image URI instead."
        detail = max(images, key=lambda item: item["imagePushedAt"])
        print("Pinned the newest tagged image:", detail.get("imageTags"))
    image_uri = registry + "/" + repository + "@" + detail["imageDigest"]
print("Image:", image_uri)


### 6. Upload source and sample inputs

- Upload the whole mlops_models directory, including model folders and configuration.
- Copy the tracked runtime/load_source.py beside it; Processing and ordinary Training run this file first.
- Put synthetic inputs under inputs/<model>/; replace this input root with your own prepared CSV folders later.

The paths were defined in cell 3. If this upload fails, you can still render
and attempt to publish the definitions in cells 7–8. This checks publication
access only; do not treat it as a runnable pipeline until uploads and job S3 access work.


In [ ]:
upload_source(session, s3_root, identity)
upload_examples(session, s3_root, model_ids)
print("Source:", source_uri)
print("Inputs:", input_root)


### 7. Build both definitions locally

- Reuse the common training/inference definitions and add the source inputs.
- ModelId selects a use case; UseHPO defaults to false and selects native tuning when true.
- InputDataRoot contains one subfolder per model.
- Training registers PendingManualApproval; inference resolves the highest Completed, Approved version.

In [ ]:
definitions = build_definitions(
    namespace, job_role_arn, image_uri, s3_root, source_uri, identity,
    region=session.region_name, models=model_ids, infrastructure=infrastructure, batch_runtime=batch_runtime,
)
WORK = ROOT / ".runs/source-deploy" / namespace
WORK.mkdir(parents=True, exist_ok=True)
for kind, body in definitions.items():
    path = WORK / (kind + ".json")
    path.write_text(json.dumps(body, indent=2) + "\n")
    print(namespace + "-" + kind, "→", " → ".join(step["Name"] for step in body["Steps"]))
    print("Definition:", path.resolve())
print("Registry groups:", [namespace + "-" + model for model in model_ids])


### 8. Publish both pipelines

- Create or update only pipelines tagged for this namespace and deployment mode.
- Read back each version and label it with the source commit; SageMaker assigns the numeric version.
- Save publication details for optional inspection. Publication is now finished.

In [ ]:
published = publish(session, namespace, pipeline_role_arn, definitions, identity)
publication = {
    "namespace": namespace, "pipelines": published, "models": model_ids,
    "source": identity, "source_uri": source_uri, "image": image_uri,
    "s3_root": s3_root, "batch_runtime": batch_runtime, "region": session.region_name,
}
path = WORK / "publication.json"
path.write_text(json.dumps(publication, indent=2) + "\n")
print("Publication:", path.resolve())


### Continue in Studio

1. Open **Pipelines → `<namespace>-training` → Create execution**. Choose `ModelId`, and name the execution `<ModelId>-training-<label>`.
   For tuning, set `UseHPO=true`, choose the trial limits, and use `<ModelId>-hpo-<label>`.
2. Inspect the evaluation report and the registered model under **Model Registry → `<namespace>-<model>`**.
3. Approve the version after reviewing it. Training finishes while approval is pending.
4. Open **`<namespace>-inference` → Create execution** and select the same model. Name it `<ModelId>-inference-<label>`.
   Inference records its exact approved package under the run's `reports/authorization`.
   It fails before scoring if none is approved.

`InputDataRoot` expects `<root>/<ModelId>/` CSV inputs.
Outputs use `<OutputDataRoot>/<ModelId>/runs/<execution-id>/`.
The optional 03G notebook can start and inspect these same executions.


The shared HPO search ranges come from `model-config.json` and must match across
the selected models. The objective metric and direction come from the selected
model at execution, so regression and classification can use different metrics.
Changing the search space requires republishing; changing UseHPO does not.
